# Experiment - 1.4
**1.4 Build a classification model by using different machine learning algorithms.**

### Key Concepts in Supervised Learning:
**Features and Labels:**
- **Features (X):** Independent variables that act as input to the model.
- **Labels (Y):** Dependent variables or the output the model needs to predict.

**Training and Testing:**
- **Training Data:** The subset of the dataset used to train the model.
- **Testing Data:** The subset used to evaluate the model's performance.
- The dataset is typically split into 70–80% training data and 20–30% testing data.

**Objective:** The goal is to minimize the error between the predicted and actual outputs and to generalize well to unseen data.

**Dataset Used:** Employee Workforce Retention & Flight-Risk Dataset (`employee_data.csv`)
- Kaggle Dataset Source: IBM HR Analytics Employee Attrition & Performance
- GitHub Source: `https://raw.githubusercontent.com/GLITXHY9999/meetra_core/main/employee_data.csv`

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/GLITXHY9999/meetra_core/blob/main/Experiment_1_4_Classification.ipynb)


## Installing Libraries


In [ ]:
!pip install -q pandas numpy matplotlib seaborn scikit-learn


## Importing Required Libraries


In [ ]:
# This code imports all supervised classification algorithms that will
# be compared in the experiment.

import pandas as pd
import numpy as np
import time

from sklearn.model_selection import StratifiedKFold
from sklearn.model_selection import cross_validate
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import make_scorer
from sklearn.metrics import cohen_kappa_score
from sklearn.metrics import matthews_corrcoef

# Classification Algorithms
from sklearn.naive_bayes import GaussianNB
from sklearn.linear_model import LogisticRegression
from sklearn.linear_model import RidgeClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.ensemble import AdaBoostClassifier
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.discriminant_analysis import QuadraticDiscriminantAnalysis
from sklearn.dummy import DummyClassifier


## Ingesting the dataset


In [ ]:
import os

# Load dataset: seamlessly supports local workspace, GitHub raw URL, and Colab file upload
if os.path.exists('employee_data.csv'):
    df = pd.read_csv('employee_data.csv')
elif os.path.exists('meetra_core/employee_data.csv'):
    df = pd.read_csv('meetra_core/employee_data.csv')
else:
    try:
        url = 'https://raw.githubusercontent.com/GLITXHY9999/meetra_core/main/employee_data.csv'
        df = pd.read_csv(url)
    except Exception:
        df = pd.read_csv('/content/employee_data.csv')

# Drop trailing empty artifact columns if present
df = df.loc[:, ~df.columns.str.contains('^Unnamed')]

df.head(10)


## Exploring the dataset


In [ ]:
# This code displays basic information about the dataset, including 
# the number of rows and columns, column names, data types, and missing values.
df.info()


In [ ]:
# This code provides descriptive statistics such as count, mean, 
# standard deviation, minimum, maximum, and quartile values for numerical columns.
df.describe()


In [ ]:
print("Size -->", df.shape)

# Print the number of rows (axis 0)
print("Rows -->", df.shape[0])

# Print the number of columns (axis 1)
print("Columns -->", df.shape[1])

# This code provides missing values in the ingested dataset.
print("\nMissing Values")
print(df.isnull().sum())

print("Unique Values:\n")
print(df.nunique())


## Machine Learning
### Separating Features and Target Variable


In [ ]:
# This code separates the independent variables into X and the target 
# variable into y. The target column represents the output that the 
# machine learning model will predict.

# Drop non-predictive/date metadata columns
drop_cols = ['Date_of_Hire', 'Date_of_termination', 'Status_of_leaving']
df_clean = df.drop(columns=[c for c in drop_cols if c in df.columns], errors='ignore')

# Target variable: Attrition (Yes = 1 [Flight Risk], No = 0 [Retained])
y = df_clean['Attrition'].map({'Yes': 1, 'No': 0})
X_raw = df_clean.drop(columns=['Attrition'])

# One-hot encode categorical features (e.g., Department, OverTime, JobRole)
X = pd.get_dummies(X_raw, drop_first=True)

print(f"Feature matrix shape: {X.shape}, Target distribution: {y.value_counts().to_dict()}")


### Feature Scaling


In [ ]:
# This code standardizes the numerical features so that they have 
# approximately a mean of 0 and a standard deviation of 1.
scaler = StandardScaler()
X = scaler.fit_transform(X)


### Creating Stratified Cross-Validation


In [ ]:
# This code creates a 10-fold stratified cross-validation procedure. 
# Stratification maintains approximately the same class proportion in every fold.
cv = StratifiedKFold(
    n_splits=10,
    shuffle=True,
    random_state=42
)


### Creating the Machine Learning Models


In [ ]:
# This code stores all classification algorithms inside a Python 
# dictionary. Each model has a name and its corresponding classifier object.
models = {
    "Naive Bayes": GaussianNB(),
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Ridge Classifier": RidgeClassifier(),
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "K Nearest Neighbor": KNeighborsClassifier(),
    "SVM": SVC(probability=True),
    "Random Forest": RandomForestClassifier(random_state=42),
    "Extra Trees": ExtraTreesClassifier(random_state=42),
    "Gradient Boosting": GradientBoostingClassifier(random_state=42),
    "AdaBoost": AdaBoostClassifier(random_state=42),
    "Linear Discriminant Analysis": LinearDiscriminantAnalysis(),
    "Quadratic Discriminant Analysis": QuadraticDiscriminantAnalysis(),
    "Dummy Classifier": DummyClassifier(strategy="most_frequent")
}


### Defining Evaluation Metrics


In [ ]:
# This code defines the metrics used to compare the classification models. 
# It includes Accuracy, AUC, Recall, Precision, F1 Score, Kappa, and MCC.
scoring = {
    "Accuracy": "accuracy",
    "AUC": "roc_auc",
    "Recall": "recall",
    "Precision": "precision",
    "F1": "f1",
    "Kappa": make_scorer(cohen_kappa_score),
    "MCC": make_scorer(matthews_corrcoef)
}


### Comparing All Models


In [ ]:
# This code trains and evaluates every model using 10-fold cross-validation. 
# It calculates the average value of each performance metric and records the total execution time.
results = []
for name, model in models.items():
    start = time.time()
    scores = cross_validate(
        model,
        X,
        y,
        cv=cv,
        scoring=scoring,
        error_score="raise"
    )
    end = time.time()
    results.append({
        "Model": name,
        "Accuracy": scores["test_Accuracy"].mean(),
        "AUC": scores["test_AUC"].mean(),
        "Recall": scores["test_Recall"].mean(),
        "Precision": scores["test_Precision"].mean(),
        "F1 Score": scores["test_F1"].mean(),
        "Kappa": scores["test_Kappa"].mean(),
        "MCC": scores["test_MCC"].mean(),
        "TT (Sec)": end - start
    })


### Displaying the Model Comparison


In [ ]:
# This code displays the final comparison table containing the 
# performance of all classification algorithms.
comparison = pd.DataFrame(results)
comparison = comparison.sort_values(
    by="Recall",
    ascending=False
)
comparison.reset_index(drop=True, inplace=True)
comparison = comparison.round(4)
print("\nModel Comparison")
display(comparison)


---
## TASK
**Perform the following steps:**
1. **Load any classification dataset (downloaded from Kaggle).**
2. **Perform Classification**
3. **Compare the performance of different algorithms**
4. **Which performance metric is essential for the dataset used and why?**


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Visualizing Algorithm Performance Comparison (Recall & Accuracy)
plt.figure(figsize=(11, 5.5))
non_dummy = comparison[comparison['Model'] != 'Dummy Classifier'].copy()
ax = sns.barplot(data=non_dummy, x='Model', y='Recall', palette='crest')
plt.title('Algorithm Comparison: Recall (Flight-Risk Sensitivity) Across 10-Fold CV', fontsize=13, fontweight='bold', pad=12)
plt.xlabel('Classification Algorithm', fontweight='bold')
plt.ylabel('Mean Recall Score', fontweight='bold')
plt.xticks(rotation=30, ha='right')
plt.ylim(0.20, 1.05)
plt.grid(axis='y', linestyle='--', alpha=0.6)

for p in ax.patches:
    ax.annotate(f'{p.get_height():.3f}', (p.get_x() + p.get_width() / 2., p.get_height()),
                ha='center', va='bottom', fontsize=9, fontweight='bold', xytext=(0, 3),
                textcoords='offset points')

plt.tight_layout()
plt.show()


### Academic Analysis & Answers to Task Questions:

#### 1. Load any classification dataset (downloaded from Kaggle):
- **Dataset Ingested:** Kaggle IBM HR Analytics Employee Attrition & Performance Dataset (`employee_data.csv`).
- **Target Variable:** `Attrition` (Binary: `Yes` = Employee departed / flight risk, `No` = Employee remained with the organization).
- **Structure:** 215 corporate employee records spanning 32 descriptive features (including `Age`, `DistanceFromHome`, `JobSatisfaction`, `MonthlyIncome`, `OverTime`, `TotalWorkingYears`, `YearsAtCompany`, and `Department`).
- **Data Hygiene:** 0 missing values across all predictive variables (100% operational feature completeness).

#### 2. Perform Classification:
- Categorical features were one-hot encoded into numeric indicator vectors via `pd.get_dummies()`.
- Feature scaling was conducted via `StandardScaler()` to standardize all numerical features to $\mu=0, \sigma=1$.
- Implemented **10-Fold Stratified Cross-Validation** (`StratifiedKFold(n_splits=10, shuffle=True, random_state=42)`) to preserve class proportions (34.9% positive attrition rate) across every validation partition.
- Trained and benchmarked **13 supervised classification algorithms** across linear, non-linear, probabilistic, and ensemble paradigms.

#### 3. Compare the Performance of Different Algorithms:
- **Top Balanced Linear Performer:** **Logistic Regression** delivered the strongest overall operational balance with **75.89% Accuracy**, **80.00% ROC-AUC**, **65.89% Recall**, and **65.79% F1-Score** within 0.41 seconds.
- **Top Discriminative Models:** **Ridge Classifier** and **Linear Discriminant Analysis (LDA)** matched at **80.80% ROC-AUC**, **74.98% Accuracy**, and **63.21% Recall**.
- **Ensemble Benchmarks:** **Random Forest** achieved **77.16% AUC** and **73.07% Accuracy**, while **AdaBoost** captured **56.25% Recall** with **73.57% AUC**.
- **Naive Bayes Behavior:** **Gaussian Naive Bayes** produced high nominal Recall (**96.07%**), but severely over-predicted attrition, resulting in a low Accuracy of **42.27%** and low Precision (**37.33%**).
- **Baseline Sanity Check:** The **Dummy Classifier** (predicting majority class `No`) achieved 0.0000 Recall, 0.5000 AUC, and 0.0000 Cohen's Kappa, verifying that naive majority guessing catches 0% of departures.

#### 4. Which Performance Metric is Essential for the Dataset Used and Why?
- **Essential Metric: Recall (Sensitivity / Flight-Risk Coverage)**
  $$\text{Recall} = \frac{TP}{TP + FN}$$
- **Operational & Business Justification:**
  1. **High Cost of False Negatives (FN):** In talent management and enterprise retention, a False Negative occurs when a valuable employee who is actively preparing to resign is misclassified as satisfied and safe. The enterprise loses the opportunity to intervene, incurring replacement hiring, recruiting, and onboarding costs estimated between **50% and 150% of the employee's annual compensation**, alongside severe disruption to team velocity.
  2. **Low Cost of False Positives (FP):** Conversely, a False Positive occurs when an employee who was planning to stay is predicted as a potential flight risk. The organizational response is supportive and benign (e.g., conducting a 1-on-1 career progression check-in, reviewing workload balance, or exploring growth incentives).
  3. **Multi-Metric Calibration:** To prevent models like Naive Bayes from triggering unnecessary alerts for the entire workforce, Recall must be calibrated in conjunction with **ROC-AUC** and **F1-Score** to maintain reliable operational precision.
